In [1]:
# Weapon class

import threading
from dataclasses import dataclass, field

# Custom exception class
class WeaponError(Exception):
    """Base class for weapon-related exceptions."""
    def __init__(self, message: str):
        super().__init__(message)

@dataclass(frozen=True)
class Weapon:
    _ammo_max: int = field(default=5)
    _ammo: int = field(default=_ammo_max)
    _damage: float = field(default=1.0)
    _loading: bool = field(default=False)
    _load_delay: float = field(default=5.0)
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    def __post_init__(self):
        if self._ammo > self._ammo_max:
            raise ValueError("Ammo cannot exceed ammo_max")

    @property
    def ammo(self) -> int:
        return self._ammo

    # @ammo.setter
    # def ammo(self, value):
    #     # return Weapon(_ammo=value, _ammo_max=self._ammo_max, _damage=self._damage, _loading=self._loading, _load_delay=self._load_delay)
    #     self._ammo = value

    @property
    def ammo_max(self) -> int:
        return self._ammo_max

    @property
    def damage(self) -> float:
        return self._damage

    @property
    def loading(self) -> bool:
        return self._loading

    @property
    def load_delay(self) -> float:
        return self._load_delay

    @property
    def is_empty(self) -> bool:
        return self.ammo == 0

    @property
    def can_reload(self) -> bool:
        return not self._loading and self._ammo < self._ammo_max

    def shot(self) -> 'Weapon':
        if self.is_empty:
            raise WeaponError("Cannot shoot: Out of ammo, current ammo: {}".format(self.ammo))
        
        with self._lock:
            return Weapon(_ammo=self._ammo - 1, _ammo_max=self._ammo_max, _damage=self._damage,
                          _loading=self._loading, _load_delay=self._load_delay)

    def reload(self) -> 'Weapon':
        if self.loading:
            raise WeaponError("Already loading. Wait for the current reload to finish.")
        
        with self._lock:
            return Weapon(_ammo=self._ammo_max, _ammo_max=self._ammo_max, _damage=self._damage,
                          _loading=True, _load_delay=self._load_delay)

    def __hash__(self):
        return hash((self._ammo, self._ammo_max, self._damage, self._loading, self._load_delay))


In [2]:
# Shield class

from dataclasses import dataclass, field
import threading
from typing import Optional

@dataclass(frozen=True)
class Shield:
    level: Optional[float] = field(default=None)  # Input shield level
    shield_max: int = field(default=5)  # Default max shield level
    repair_delay: int = field(default=5)   # Default repair delay
    _is_repairing: threading.Event = field(default_factory=threading.Event, init=False)
    lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    def __post_init__(self):
        # Normalize level to be >= 0 and <= shield_max
        if self.level is None:
            object.__setattr__(self, 'level', self.shield_max)
        else:
            normalized_level = min(max(self.level, 0), self.shield_max)
            object.__setattr__(self, 'level', normalized_level)

        object.__setattr__(self, '_is_repairing', False)  # Initialize repairing status

    @property
    def is_repairing(self) -> bool:
        """Check if the shield is currently being repaired."""
        with self.lock:
            return self._is_repairing

    def repair_shield(self) -> None:
        """Initiate a repair of the shield after a delay."""
        with self.lock:
            current_level = self.level if self.level is not None else 0
            if current_level < self.shield_max and not self._is_repairing:
                object.__setattr__(self, '_is_repairing', True)
                threading.Timer(self.repair_delay, self._finish_repair).start()

    def _finish_repair(self) -> None:
        """Finish repairing the shield."""
        with self.lock:
            object.__setattr__(self, 'level', self.shield_max)
            object.__setattr__(self, '_is_repairing', False)

    def damage_shield(self, damage: float) -> 'Shield':
        """Apply damage to the shield, returning a new Shield instance with updated level."""
        with self.lock:
            current_level = self.level if self.level is not None else 0
            new_level = max(current_level - damage, 0)
            return Shield(level=new_level, shield_max=self.shield_max, repair_delay=self.repair_delay)

    def __eq__(self, other: object) -> bool:
        """Check if two shields are equal based on their levels and max values."""
        if not isinstance(other, Shield):
            return NotImplemented
        return (self.level == other.level and 
                self.shield_max == other.shield_max)

    def __hash__(self) -> int:
        """Return the hash based on shield level and max."""
        return hash((self.level, self.shield_max))

    def __str__(self) -> str:
        return (f"Shield Level: {self.level}/{self.shield_max}, "
                f"Repairing: {self.is_repairing}")

    def __lt__(self, other: 'object') -> bool:
        """Compare shields based on their shield levels."""
        if not isinstance(other, Shield):
            return NotImplemented
        s = self.level if self.level is not None else 0
        o = other.level if other.level is not None else 0
        return s < o
        

In [3]:
# FuelTank class

from dataclasses import dataclass, field
from math import inf


@dataclass(frozen=True)
class FuelTank:
    level: float = field(default=-inf)
    cost: float = field(default=0.25)
    volume: float = field(default=100)

    def __post_init__(self):
        if self.level > self.volume:
            raise ValueError("Fuel tank level cannot be greater than volume")
        if self.level < 0:
            object.__setattr__(self, "level", self.volume)
        if self.cost <= 0 or self.volume <= 0:
            raise ValueError("Cost and volume must be positive.")

    def drop_fuel(self, distance: float):
        new_level = self.level - (distance * self.cost)
        if new_level < 0:
            raise ValueError("Fuel tank level cannot be negative")
        return FuelTank(level=new_level, cost=self.cost, volume=self.volume)

    def refuel(self):
        # takes time can stop mid-way
        ...
        if self.is_full():
            raise ValueError("Fuel tank is full")

        return FuelTank(level=self.volume, cost=self.cost, volume=self.volume)

    def is_empty(self):
        return self.level == 0

    def is_full(self):
        return self.level == self.volume

In [ ]:
# Robot class

from dataclasses import dataclass, field
from enum import Enum
from typing import Optional

# from src.main.world_objects.robot_objects.fuel_tank import FuelTank
# from src.main.world_objects.robot_objects.position import Position
# from src.main.world_objects.robot_objects.degrees import Degrees
# from src.main.world_objects.robot_objects.shield import Shield
# from src.main.world_objects.robot_objects.weapon import Weapon, WeaponError


class RobotType(Enum):
    SCOUT = "scout"
    SNIPER = "sniper"
    TANK = "tank"
    ASSAULT = "assault"
    SUPPORT = "support"


@dataclass
class Robot:
    name: str = field(default="bot")
    position: Position = field(default_factory=lambda: Position(0, 0))
    direction: Degrees = field(default_factory=lambda: Degrees(0))
    shield: Shield = field(default_factory=lambda: Shield(shield_max=5))
    weapon: Weapon = field(default_factory=lambda: Weapon(_ammo=5))
    tank: FuelTank = field(default_factory=lambda: FuelTank(volume=50))
    robot_type: RobotType = field(default=RobotType.SUPPORT)

    def __post_init__(self):
        self._set_attributes_based_on_type()

    def move_forward(self, nr_steps: int) -> bool:
        return self._update_position(nr_steps, forward=True)

    def move_backward(self, nr_steps: int) -> bool:
        return self._update_position(nr_steps, forward=False)

    def turn_left(self, degrees: float = 90) -> None:
        return Robot(name=self.name, position=self.position, direction=self.direction.turn_left(degrees), robot_type=self.robot_type, tank=self.tank)
        
    def turn_right(self, degrees: float = 90) -> None:
        return Robot(name=self.name, position=self.position, direction=self.direction.turn_right(degrees), robot_type=self.robot_type, tank=self.tank)

    def damage_shield(self, damage: float) -> None:
        return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank, shield=self.shield.damage_shield(damage))

    def repair_shield(self) -> None:
        return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank, shield=self.shield.repair_shield())
        
    def shoot(self) -> None:
        try:
            return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank, weapon=self.weapon.shot())
        except WeaponError as e:
            print(f"{e}")

    def reload(self) -> None:
        try:
            return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank, weapon=self.weapon.reload())
        except WeaponError as e:
            print(e)

    def refuel(self) -> None:
        try:
            return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank.refuel(), weapon=self.weapon)
        except ValueError as e:
            print(e)

    def shield_level(self) -> Optional[int]:
        return self.shield.level

    def tank_level(self) -> float:
        return self.tank.level

    def _update_position(self, nr_steps: int, forward: bool) -> bool:
        steps = nr_steps if forward else -nr_steps
        
        if self._drop_fuel(nr_steps): 
            return Robot(name=self.name, position=self.position.move(self.direction, steps), direction=self.direction,robot_type=self.robot_type, tank=self.tank)
        return self # return the new robot with the updated position since all objects have to be hashable (frozen dataclasses)
        

    def _drop_fuel(self, steps: int):
        try:
            return Robot(name=self.name, position=self.position, direction=self.direction, robot_type=self.robot_type, tank=self.tank.drop_fuel(distance=steps))
        except ValueError:
            print("not enough fuel")
            return self

    
    def _set_attributes_based_on_type(self):
        # Define attributes based on the RobotType Enum
        type_attributes = {
            RobotType.SCOUT: (1, 1, 1, 1, 1),
            RobotType.SNIPER: (2, 2, 2, 2, 2),
            RobotType.TANK: (3, 3, 3, 3, 3),
            RobotType.ASSAULT: (4, 4, 4, 4, 4),
            RobotType.SUPPORT: (5, 5, 5, 5, 5),
        }

        # Normalize robot_type first
        if isinstance(self.robot_type, str):
            try:
                self.robot_type = RobotType(self.robot_type.lower())
            except ValueError:
                allowed = [t.value for t in RobotType]
                print(f"Invalid robot type: '{self.robot_type}'. Must be one of: {allowed}")
                self.robot_type = RobotType.SUPPORT

        # Look up attributes based on the robot's type
        attributes = type_attributes.get(self.robot_type)
        if attributes:
            shot_damage, ammo_max, shield_max, repair_delay, reload_delay = attributes
            self.weapon = Weapon(
                _ammo=ammo_max, _load_delay=reload_delay, _damage=shot_damage, _ammo_max=ammo_max)
            self.shield = Shield(shield_max=shield_max,
                                 repair_delay=repair_delay)

    def __str__(self):
        return (
            f"Name: {self.name}, Position: {self.position}, Direction: {self.direction.angle}, "
            f"Shield Level: {self.shield_level()}, Ammo: {self.weapon.ammo}, Fuel Level: {self.tank_level()}, Type: {self.robot_type.value}"
        )
